In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('AAPL.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print("Форма:", df.shape)
print()
print(df.head())
print()
print(df.dtypes)

Форма: (10409, 7)

        Date      Open      High       Low     Close  Adj Close     Volume
0 1980-12-12  0.128348  0.128906  0.128348  0.128348   0.100323  469033600
1 1980-12-15  0.122210  0.122210  0.121652  0.121652   0.095089  175884800
2 1980-12-16  0.113281  0.113281  0.112723  0.112723   0.088110  105728000
3 1980-12-17  0.115513  0.116071  0.115513  0.115513   0.090291   86441600
4 1980-12-18  0.118862  0.119420  0.118862  0.118862   0.092908   73449600

Date         datetime64[us]
Open                float64
High                float64
Low                 float64
Close               float64
Adj Close           float64
Volume                int64
dtype: object


In [5]:
# Дневная доходность
df['Return'] = df['Close'].pct_change()

# Логарифмическая доходность
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Скользящие средние
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

# Экспоненциальное скользящее среднее
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# Волатильность
df['Volatility_20'] = df['Return'].rolling(window=20).std()

# Расстояние цены от SMA_20
df['Close_vs_SMA20'] = (df['Close'] - df['SMA_20']) / df['SMA_20']

print(df[['Return', 'Log_Return', 'SMA_20', 'EMA_20', 'Volatility_20']].head(25))

      Return  Log_Return    SMA_20    EMA_20  Volatility_20
0        NaN         NaN       NaN  0.128348            NaN
1  -0.052171   -0.053581       NaN  0.127710            NaN
2  -0.073398   -0.076231       NaN  0.126283            NaN
3   0.024751    0.024450       NaN  0.125257            NaN
4   0.028992    0.028580       NaN  0.124648            NaN
5   0.061029    0.059239       NaN  0.124788            NaN
6   0.048669    0.047522       NaN  0.125499            NaN
7   0.042199    0.041333       NaN  0.126674            NaN
8   0.052628    0.051290       NaN  0.128428            NaN
9   0.092309    0.088294       NaN  0.131290            NaN
10  0.014084    0.013985       NaN  0.134092            NaN
11 -0.024304   -0.024604       NaN  0.136256            NaN
12 -0.028468   -0.028881       NaN  0.137788            NaN
13  0.010988    0.010928       NaN  0.139334            NaN
14 -0.021738   -0.021977       NaN  0.140413            NaN
15 -0.044448   -0.045466       NaN  0.14

In [6]:
!pip install ta 

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29496 sha256=48f19c3de915de8c020991cc0db8485566225b3d066f31fc8faf1c1b6ff47137
  Stored in directory: c:\users\ае\appdata\local\pip\cache\wheels\e3\3a\ee\4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta


In [7]:
import ta

# RSI — индекс относительной силы (14 дней)
df['RSI_14'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(close=df['Close'])
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()

# Bollinger Bands
bb = ta.volatility.BollingerBands(close=df['Close'], window=20)
df['BB_upper'] = bb.bollinger_hband()
df['BB_lower'] = bb.bollinger_lband()

# ATR — средний истинный диапазон
df['ATR_14'] = ta.volatility.AverageTrueRange(
    high=df['High'], low=df['Low'], close=df['Close'], window=14
).average_true_range()

print(df[['RSI_14', 'MACD', 'BB_upper', 'ATR_14']].head(30))

       RSI_14      MACD  BB_upper    ATR_14
0         NaN       NaN       NaN  0.000000
1         NaN       NaN       NaN  0.000000
2         NaN       NaN       NaN  0.000000
3         NaN       NaN       NaN  0.000000
4         NaN       NaN       NaN  0.000000
5         NaN       NaN       NaN  0.000000
6         NaN       NaN       NaN  0.000000
7         NaN       NaN       NaN  0.000000
8         NaN       NaN       NaN  0.000000
9         NaN       NaN       NaN  0.000000
10        NaN       NaN       NaN  0.000000
11        NaN       NaN       NaN  0.000000
12        NaN       NaN       NaN  0.000000
13  69.708734       NaN       NaN  0.005700
14  64.735466       NaN       NaN  0.005532
15  56.111860       NaN       NaN  0.005615
16  49.591320       NaN       NaN  0.005652
17  46.922193       NaN       NaN  0.005448
18  53.873578       NaN       NaN  0.005617
19  52.729429       NaN  0.167133  0.005295
20  47.808888       NaN  0.167164  0.005276
21  48.385225       NaN  0.16680

In [ ]:
# Добавляем День недели и месяц

In [8]:
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()

print(df[['Date', 'Day_of_Week', 'Month']].head(10))
print()
print(df['Day_of_Week'].value_counts())
print()
print(df['Month'].value_counts())

        Date Day_of_Week     Month
0 1980-12-12      Friday  December
1 1980-12-15      Monday  December
2 1980-12-16     Tuesday  December
3 1980-12-17   Wednesday  December
4 1980-12-18    Thursday  December
5 1980-12-19      Friday  December
6 1980-12-22      Monday  December
7 1980-12-23     Tuesday  December
8 1980-12-24   Wednesday  December
9 1980-12-26      Friday  December

Day_of_Week
Tuesday      2134
Wednesday    2133
Thursday     2094
Friday       2078
Monday       1970
Name: count, dtype: int64

Month
March        916
August       907
October      905
December     880
June         878
July         868
January      866
May          865
April        848
November     838
September    833
February     805
Name: count, dtype: int64


In [ ]:
# Удаляем NaN

In [9]:
print("До очистки:", df.shape)
df = df.dropna().reset_index(drop=True)
print("После очистки:", df.shape)

До очистки: (10409, 23)
После очистки: (10360, 23)


In [ ]:
# потеряли 49, переживем)))

In [ ]:
# проверка

In [10]:
print("Форма:", df.shape)
print()
print("Столбцы:", df.columns.tolist())
print()
print("Пропусков:", df.isna().sum().sum())
print()
df.describe().round(3)

Форма: (10360, 23)

Столбцы: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Return', 'Log_Return', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Close_vs_SMA20', 'RSI_14', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'ATR_14', 'Day_of_Week', 'Month']

Пропусков: 0



,Date,Open,High,Low,Close,Adj Close,Volume,Return,Log_Return,SMA_10,...,SMA_50,EMA_20,Volatility_20,Close_vs_SMA20,RSI_14,MACD,MACD_signal,BB_upper,BB_lower,ATR_14
count,10360,10360.000,10360.000,10360.000,10360.000,10360.000,1.036000e+04,10360.000,10360.000,10360.000,...,10360.000,10360.000,10360.000,10360.000,10360.000,10360.000,10360.000,10360.000,10360.000,10360.000
mean,2001-08-30 22:09:13.204633,14.025,14.178,13.874,14.032,13.413,3.335353e+08,0.001,0.001,13.960,...,13.643,13.882,0.025,0.008,53.141,0.111,0.111,14.716,13.049,0.331
min,1981-02-24 00:00:00,0.050,0.050,0.049,0.049,0.038,0.000000e+00,-0.519,-0.731,0.053,...,0.058,0.056,0.005,-0.569,15.346,-4.665,-3.805,0.061,0.042,0.001
25%,1991-05-21 18:00:00,0.286,0.292,0.279,0.286,0.238,1.259524e+08,-0.013,-0.013,0.286,...,0.286,0.285,0.017,-0.031,43.634,-0.006,-0.006,0.316,0.252,0.011
50%,2001-08-21 12:00:00,0.473,0.482,0.464,0.475,0.391,2.216474e+08,0.000,0.000,0.472,...,0.464,0.465,0.023,0.009,52.962,0.004,0.004,0.513,0.427,0.020
75%,2011-12-07 06:00:00,14.325,14.489,14.163,14.354,12.327,4.144888e+08,0.015,0.015,14.321,...,14.013,14.208,0.031,0.048,62.668,0.053,0.051,15.024,13.330,0.323
max,2022-03-24 00:00:00,182.630,182.940,179.120,182.010,181.778,7.421641e+09,0.332,0.287,178.332,...,172.664,174.826,0.129,0.617,91.503,7.512,6.896,185.173,169.905,5.596
std,NaN,30.225,30.572,29.891,30.248,29.968,3.395278e+08,0.028,0.029,30.071,...,29.298,29.876,0.013,0.070,13.066,0.683,0.645,31.675,28.135,0.773


In [11]:
df.to_csv('AAPL_features.csv', index=False)
print("Сохранено:", df.shape)

Сохранено: (10360, 23)


In [12]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
print(df['Target'].value_counts())

Target
0    5222
1    5138
Name: count, dtype: int64


In [13]:
df = df.dropna().reset_index(drop=True)
df.to_csv('AAPL_features.csv', index=False)
print("Сохранено:", df.shape)

Сохранено: (10360, 24)


In [14]:
# удаляем ложный таргет
df = df.iloc[:-1].reset_index(drop=True)
df.to_csv('AAPL_features.csv', index=False)
print("Сохранено:", df.shape)

Сохранено: (10359, 24)
